In [3]:
import os
import time
from libtmux import Server

In [4]:
original_model_path = "/data/shared/charlie/GLM4.7-sft-mix"
huggingface_path = "huggingface_10000/"
model_assets_path = "model_assets/"
nodes = [
    "research-secure-08"
]

# If a teammate already launched sglang elsewhere, point this at that
# OpenAI-compatible endpoint and keep local tmux launch disabled.
sglang_endpoint = "http://research-secure-08:30100/v1"
sglang_api_key = "sk-dummy-key-for-local-model"
launch_sglang_in_tmux = False

# openai/Qwen/Qwen3-Coder-sft-mix-14000
# rebench-verify

# tmux -S /data/jisenli2/.tmux attach -t r2egym
# tmux -S /data/jisenli2/.tmux-secure-08 attach -t r2egym

In [5]:
import os
from libtmux import Server

tmux_socket_path = "/data/jisenli2/.tmux-secure-08"
server = Server(socket_path=tmux_socket_path)

sessions = server.sessions
print(f"Found {len(sessions)} sessions to delete")

for session in sessions:
    try:
        session_name = session.name
        session.kill()
        print(f"✅ Session '{session_name}' deleted successfully")
    except Exception as e:
        print(f"❌ Error deleting session: {e}")

print("All sessions deleted!")

Found 3 sessions to delete
✅ Session 'docker' deleted successfully
✅ Session 'r2egym' deleted successfully
✅ Session 'sglang' deleted successfully
All sessions deleted!


# sglang

In [6]:
# connect to the docker server
session_name = "sglang"
tmux_socket_path = "/data/jisenli2/.tmux-secure-08"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [7]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Successfully connected to research-secure-08


# docker

In [8]:
# connect to the docker server
session_name = "docker"
tmux_socket_path = "/data/jisenli2/.tmux-secure-08"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [9]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Successfully connected to research-secure-08


In [10]:
CMD = """
sudo usermod -aG docker $USER
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 08


In [11]:
CMD = """
newgrp docker
"""

# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")


  ✅ Command sent to 08


In [12]:
CMD = """conda activate r2egym
cd /data/jisenli2/R2E-Gym/scripts
python docker_storage_monitor.py
"""
# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 08


# r2egym

In [13]:
# connect to the docker server
session_name = "r2egym"
tmux_socket_path = "/data/jisenli2/.tmux-secure-08"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [14]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Successfully connected to research-secure-08


In [15]:


CMD = """
sudo usermod -aG docker $USER
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")



  ✅ Command sent to 08


In [16]:
CMD = """
newgrp docker
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 08


In [ ]:
# Send vllm serve command to each window
# Only do it when disk is full
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys('sudo docker system prune -a --volumes --force', enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 08


In [17]:
import datasets
dataset = datasets.load_dataset("togethercomputer/R2E-Gym-Openhands-SWE-Bench-Verified", split="test")
# dataset = datasets.load_dataset("togethercomputer/R2E-Gym-Openhands-SWE-Bench-Pro", split="train")

In [18]:
dataset

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'parsed_commit', 'run_tests', 'docker_image'],
    num_rows: 500
})

In [19]:
CMD_TEMPLATE = """
conda activate r2egym
cd /data/jisenli2/R2E-Gym

python src/r2egym/run/eval_openhands.py \
  --max_workers 24 \
  --dataset_num_shards {num_nodes} \\
  --dataset_shard_index {node_idx} \\
  --dataset "togethercomputer/R2E-Gym-Openhands-SWE-Bench-Verified" \
  --split "test" \
  --llm_name "openai/GLM4.7" \
  --base_url "http://localhost:30100/v1" \
  --api_key "sk-dummy-key-for-local-model" \
  --use_fn_calling True \
  --exp_name glm4.7-fp8-kv16-baseline \
  --output_dir /data/jisenli2/kv_rotation/swe_bench_verified_results/glm4.7-fp8-kv16-baseline \
  --temperature 0.7 \
  --top_p 1.0 \
  --max_steps 100 \
  --backend "docker" \
  --max_reward_calc_time 1200 \
  --env_mode "swebench" \
  --max_output_tokens 131072 \
  --pass_n {pass_n}
"""

# CMD_TEMPLATE = """
# conda activate r2egym
# cd /data/jisenli2/R2E-Gym-swepro/R2E-Gym

# python src/r2egym/run/eval_openhands.py \
#   --max_workers 24 \
#   --dataset_num_shards {num_nodes} \\
#   --dataset_shard_index {node_idx} \\
#   --dataset "togethercomputer/R2E-Gym-Openhands-SWE-Bench-Pro" \
#   --split "train" \
#   --llm_name "openai/Qwen/Qwen3-Coder-sft-mix-25000_no_penalty" \
#   --base_url "http://localhost:30000/v1" \
#   --api_key "sk-dummy-key-for-local-model" \
#   --use_fn_calling True \
#   --exp_name swebench-python-qwen3-coder-32b-sft-mix-25000_no_penalty \
#   --output_dir ./swebench-python-qwen3-coder-32b-sft-mix-25000_no_penalty \
#   --temperature 0.7 \
#   --top_p 1.0 \
#   --max_steps 100 \
#   --backend "docker" \
#   --max_reward_calc_time 1200 \
#   --env_mode "swebench" \
#   --max_output_tokens 131072 \
#   --pass_n {pass_n}
# """

pass_n = 1

# Calculate dataset sharding for 3 nodes
total_examples = len(dataset)
num_nodes = len(nodes)

# Create commands for each node
commands = []
for node_idx in range(num_nodes):
    cmd = CMD_TEMPLATE.format(node_idx=node_idx, num_nodes=num_nodes, pass_n=pass_n)
    commands.append(cmd)
    print(f"Node {node_idx}: num_nodes={num_nodes}")


# Send vllm serve command to each window
for idx, window in enumerate(session.windows):
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(commands[idx], enter=True)
    print(f"  ✅ Command sent to {window_name}")

Node 0: num_nodes=1
  ✅ Command sent to 08


# results collects

In [9]:
import os
import tqdm
import json
import glob
import tqdm
import datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from typing import List, Dict, Optional
def print_reward_files(path):
    reward_files = glob.glob(os.path.join(path, "**", "reward.json"))

    from concurrent.futures import ThreadPoolExecutor, as_completed
    import json

    def read_reward_file(file):
        with open(file, "r") as f:
            return json.load(f)

    with ThreadPoolExecutor(max_workers=64) as executor:
        futures = {executor.submit(read_reward_file, file): file for file in reward_files}
        all_data = []
        for future in tqdm.tqdm(as_completed(futures), total=len(reward_files), desc="Reading reward files"):
            all_data.append(future.result())

    print(len(all_data))
    rewards = [data["reward"] for data in all_data]
    print(np.mean(rewards))
    # 1 / (9000 * 4 / len(reward_files))
    # 4500 * 8

In [10]:
path = "/data/jisenli2/kv_rotation/swe_bench_verified_results/glm4.7-fp8-kv16-baseline/togethercomputer_R2E-Gym-Openhands-SWE-Bench-Verified/OpenhandsAgent/openai_GLM4_7_max100_temp0.7_topp1.0/run1"
print_reward_files(path)

Reading reward files: 100%|██████████| 228/228 [00:00<00:00, 4916.77it/s]

228
0.6403508771929824
